# 00. Brownfield Baseline and Azure Bootstrap

This notebook is the entry point for `v0.1: Brownfield Baseline, Repository Bootstrap, and Azure Foundation`.

It covers ten things:

1. What the old backend does
2. How a request flowed through it
3. Which components are worth reusing
4. Which components are obsolete
5. Why Azure SQL with AdventureWorksLT replaces the local Chinook database
6. Why provisioning and seeding are different concerns
7. The target Azure foundation architecture
8. The Azure prerequisites to validate before provisioning
9. What v0.1 will and will not provision
10. The before and after measurements that define progress

No agent is built here, and no LangGraph code is migrated. This release changes the substrate so that later
releases modernize the agent on a platform that already handles identity, least privilege, monitoring, and
lifecycle correctly.

Every reusable implementation lives in `src/enterprise_agents_on_foundry/setup/`. This notebook imports it.
Nothing of substance is defined inline, so the same code runs in the notebook, in the scripts, and in the tests.

## Setup

Run `uv sync` before opening this notebook, and select the `.venv` interpreter as the kernel.

In [7]:
from enterprise_agents_on_foundry.setup import (
    MeasurementSet,
    assert_read_only_sql,
    check_all_prerequisites,
    check_model_availability,
    check_subscription,
    get_signed_in_principal_id,
    is_read_only_sql,
    load_settings,
    repository_root,
    resolve_database_target,
    time_operation,
)

ROOT = repository_root()
settings = load_settings()
measurements = MeasurementSet()

print(f"Repository : {ROOT}")
print(f"Dataset    : {settings.dataset_variant}")
print(f"Region     : {settings.azure_location}")
print(f"Provisioned: {settings.is_provisioned}")

Repository : C:\Users\shchitt\Downloads\Projects\enterprise-agents-workspace\enterprise-agents-on-foundry
Dataset    : adventureworks-lt
Region     : westus3
Provisioned: True


## 1. What the old backend does

The reference project is at `../langgraph-agents-on-azure/backend` on branch `shiva-cicd-branch`. It is treated
as read-only and is never modified.

It is a LangGraph Text-to-SQL agent behind a FastAPI service, answering natural language questions over the
Chinook sample database. The pieces are:

| Path | Role |
| --- | --- |
| `main.py` | Eight lines, `uvicorn.run(app, host="0.0.0.0", port=80)` with the dev port commented out |
| `api/service.py` | FastAPI with `/health`, `/sql-invoke`, `/sql-stream` |
| `api/schema.py` | Pydantic request and response models with LangChain conversion |
| `api/settings.py` | `BaseSettings` whose defaults are evaluated at class definition |
| `agents/sql_agent/sql_agent.py` | `AzureSqlAgent`, a hand-wired seven-node `StateGraph` |
| `agents/sql_agent/sql_agent_postgres.py` | A near-duplicate of the above |
| `agents/sql_agent/tools.py` | Builds the model and the database connection at import time |
| `agents/sql_agent/prompts.py` | System prompt asking for SQLite syntax and no DML |
| `agents/checkpointer.py` | SQLite checkpointer helpers that swallow errors and return `None` |
| `agents/tracing.py` | OpenTelemetry and Prompty tracing, duplicated in `notebooks/` |
| `agents/visualization_agent/` | A complete Plotly agent that nothing imports |
| `evaluations/evaluation.py` | Evaluators pointed at a hardcoded public IP address |

The full inventory with sizes and findings is in `docs/current-state/backend-inventory.md`.

## 2. How a request flowed through it

A call to `POST /sql-invoke` took nine steps.

```text
validate UserInput
  -> print full kwargs including the user question to stdout
  -> invoke graph with a thread id, loading prior turns from the checkpointer
  -> list_tables_ai_call emits a fabricated tool call id "tool_abcd123"
  -> list_tables_tool -> get_relevant_schema -> get_schema_tool
  -> query_agent generates SQL under a prompt asking for no DML
  -> tool_check routes to execute_query (loop on error) or generate_answer
  -> delete_intermediate_messages, then trim to the last 6 messages
  -> ChatMessage.from_langchain
```

Five properties of that flow matter.

Safety is advisory. Step 6 forbids DML in prose only. Nothing validates the SQL and the database principal has
full rights over the file, so the only barrier is model compliance.

Errors are invisible. `db.run_no_throw(query)` returns error text as an ordinary result, so a failed query and
an empty result are indistinguishable without parsing prose.

Import has side effects. `tools.py` builds `AzureChatOpenAI` and `SQLDatabase.from_uri("sqlite:///chinook.db")`
at module scope, so importing it requires credentials and a working directory containing the database file.

Context management is a magic number. Trimming to six messages is unrelated to the model context window.

State is not durable. The checkpointer writes to a file inside the container, so restarting loses history and
two replicas diverge.

## 3. Which components are worth reusing

Not everything in the legacy project is wrong. Four things are carried forward.

The message boundary in `api/schema.py`. `UserInput`, `StreamInput`, `AgentResponse`, and `ChatMessage` keep
framework types out of the public contract, and `from_langchain` / `to_langchain` are the right shape.

The root span concept in `agents/tracing.py`. Wrapping a request in one span with the turn as its unit of work
is correct. The environment variable name is not, and that is fixed rather than the idea.

The progressive notebook format. Six numbered notebooks that each add one concept is exactly the teaching
structure this repository continues.

The evaluation dataset shape. Question and ground-truth pairs with prose answers work. The Chinook content does
not survive the dataset change, but the structure does, extended in `evals/datasets/legacy-baseline.jsonl`.

## 4. Which components are obsolete

Nine things are deliberately not carried forward.

The LangGraph 0.2 hand-wired graph, including the fabricated `"tool_abcd123"` tool call, which worked around an
API shape that has since been replaced.

Module-level model and database instantiation, which makes the code untestable and working-directory dependent.

The embedded SQLite Chinook database, and every committed `.db` file. There were five, totalling roughly 5 MB,
three of them containing real conversation state.

Dual SQLite and PostgreSQL checkpointers, together with the duplicated `sql_agent_postgres.py`. Two copies of
the same graph diverging by 0.8 KB is the clearest indicator of copy-paste drift in the repository.

API key authentication and password-bearing environment contracts.

`requirements.txt` with 31 exact pins, no lockfile, no groups, five unused packages, and one package imported
but never declared.

`allow_origins=["*"]` combined with `allow_credentials=True`, a pairing browsers reject.

Hardcoded public IP addresses in evaluation code.

Manual Docker, ACR, Web App, and AKS steps as the initial deployment model.

The reasoning for each is in `docs/current-state/modernization-risks.md`.

## 5. Why Azure SQL with AdventureWorksLT replaces local Chinook

The dataset is not the point. The substrate is.

A local SQLite file has no network boundary, so connection handling, timeouts, retries, and transient faults
never appear. It has no identity, so authentication and least privilege cannot be demonstrated. It has no
server-side permission model, so the prompt-only DML restriction has no enforceable counterpart. And it is a
local file, so a hosted agent running in Azure cannot reach it at all.

Later releases cover hosted agents, observability, performance, token economics, and governance. Every one of
those needs a data source that behaves like a real one.

AdventureWorksLT rather than full AdventureWorks OLTP, because the lightweight variant has enough surface for
joins, aggregates, grouping, ordering, and date filtering while staying small enough to hold in mind while
reading a generated query.

The decision record is `docs/adr/002-use-azure-sql-adventureworks-lt.md`.

### The safety change this enables

The legacy system asked the model not to write. This release enforces it twice, independently.

The database principal holds `db_datareader` and is explicitly denied `INSERT`, `UPDATE`, `DELETE`, `ALTER`, and
`EXECUTE`, so a destructive statement fails at the server regardless of what produced it.

Separately, `assert_read_only_sql` rejects anything that is not a single `SELECT` or `WITH` statement. It strips
comments before matching so a keyword cannot hide behind one, and it tracks string literals so a semicolon
inside a value is not read as a statement separator.

Neither control depends on the other, and neither depends on the prompt.

In [8]:
examples = [
    "SELECT TOP (10) Name FROM SalesLT.Product",
    "SELECT * FROM SalesLT.Customer WHERE CompanyName = 'Bikes; Parts'",
    "DROP TABLE SalesLT.Product",
    "SELECT 1; DROP TABLE SalesLT.Product",
    "/* SELECT */ DELETE FROM SalesLT.Product",
    "EXEC sp_who",
]

for sql in examples:
    verdict = "allowed" if is_read_only_sql(sql) else "rejected"
    print(f"{verdict:>8}  {sql}")

 allowed  SELECT TOP (10) Name FROM SalesLT.Product
 allowed  SELECT * FROM SalesLT.Customer WHERE CompanyName = 'Bikes; Parts'
rejected  DROP TABLE SalesLT.Product
rejected  SELECT 1; DROP TABLE SalesLT.Product
rejected  /* SELECT */ DELETE FROM SalesLT.Product
rejected  EXEC sp_who


In [9]:
# Rejection reasons are specific, so a failure explains itself.
for sql in ["SELECT 1; DROP TABLE SalesLT.Product", "UPDATE SalesLT.Product SET ListPrice = 0", "-- nothing"]:
    try:
        assert_read_only_sql(sql)
    except ValueError as error:
        print(f"{sql!r}\n  -> {error}\n")

'SELECT 1; DROP TABLE SalesLT.Product'
  -> Query contains 2 statements; only a single statement is allowed.

'UPDATE SalesLT.Product SET ListPrice = 0'
  -> Query must begin with SELECT or WITH; found 'UPDATE'.

'-- nothing'
  -> Query contains no executable statement.



## 6. Why provisioning and seeding are different concerns

Conflating these is what made the legacy setup hard to reason about, so v0.1 separates them explicitly.

Provisioning the server and database is a control-plane operation. It is declarative, idempotent, reviewed as
infrastructure, and lives in Bicep. The `Microsoft.Sql/servers/databases` resource exposes a `sampleName`
property whose enumeration includes `AdventureWorksLT`, so the database arrives populated with no data-plane
step at all.

Populating and permissioning data is a data-plane operation. It needs a database connection, an authenticated
principal, and a safety switch, so it lives in `scripts/bootstrap_database.py`.

Validating connectivity and inspecting the schema is a third concern. It never modifies anything, so it runs by
default and needs no switch.

The practical consequence is the safety switch. `scripts/bootstrap_database.py` resolves and prints its target
before doing anything, and refuses to modify it unless `ALLOW_DATABASE_BOOTSTRAP=true`.

In [10]:
from enterprise_agents_on_foundry.setup import require_bootstrap_allowed

print(f"ALLOW_DATABASE_BOOTSTRAP = {settings.allow_database_bootstrap}")

try:
    require_bootstrap_allowed(settings)
    print("Bootstrap is permitted.")
except PermissionError as error:
    print(f"Refused: {error}")

ALLOW_DATABASE_BOOTSTRAP = True
Bootstrap is permitted.


In [11]:
# The target is validated before any connection is attempted, so a misconfiguration
# fails with a readable message instead of an ODBC error.
if settings.is_provisioned:
    target = resolve_database_target(settings)
    print(f"Target: {target.display}")
    print(f"Connection string (no password, ever):\n  {target.odbc_connection_string()}")
else:
    print("Not provisioned yet. Missing:")
    for name in settings.missing_provisioning_outputs():
        print(f"  {name}")

Target: sql-eaof-dev-wgi4fh.database.windows.net/AdventureWorksLT (auth=entra)
Connection string (no password, ever):
  Driver={ODBC Driver 18 for SQL Server};Server=tcp:sql-eaof-dev-wgi4fh.database.windows.net,1433;Database=AdventureWorksLT;Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;


## 7. Target Azure foundation architecture

Eleven resources in one resource group, all created by `infra/main.bicep`.

| Resource | Purpose |
| --- | --- |
| Foundry account | AI Services account with `allowProjectManagement: true` and `disableLocalAuth: true` |
| Foundry project | Child project scoping agents and connections |
| Model deployment | One inexpensive chat model on Global Standard |
| Azure SQL server | Logical server, `azureADOnlyAuthentication: true`, no password parameter exists |
| Azure SQL database | Serverless `GP_S_Gen5`, AdventureWorksLT, auto-pause after 60 idle minutes |
| Log Analytics workspace | Telemetry store, 30-day retention |
| Application Insights | Workspace-based, `DisableLocalAuth: true` |
| Key Vault | RBAC authorization rather than access policies |
| Managed identity | User-assigned workload identity |
| Container registry | Optional, disabled in v0.1 |

There are no keys, connection strings, or passwords anywhere in the access model. Five control-plane role
assignments connect the deployer and the managed identity to Foundry, Key Vault, and Application Insights.

Database access is deliberately absent from those five. Azure role-based access control governs the SQL
resource, not the rows inside it, so read access is granted in-database instead. A test asserts that the RBAC
module never references `Microsoft.Sql`, so the two mechanisms cannot be confused.

Every non-obvious property was verified against the live resource provider rather than copied from an example.
The evidence table is in `docs/architecture/v0.1-azure-foundation.md`.

## 8. Azure prerequisites to validate

Six things must be true before provisioning. Checking them takes seconds; discovering them mid-deployment does
not.

In [12]:
report = check_all_prerequisites()

for name, status, version in report.as_rows():
    print(f"{name:<8} {status:<9} {version}")

print()
print("All required tools present." if report.ok else f"Missing: {', '.join(report.missing_required)}")

python   ok        3.12.11
git      ok        git version 2.55.0.windows.3
uv       ok        uv 0.8.3 (7e78f54e7 2025-07-24)
az       ok        azure-cli                         2.69.0 *
azd      ok        azd version 1.28.1 (commit 3cf6db5881d8b24bb497e1470a972f4e28eb0256) (stable)
bicep    ok        Bicep CLI version 0.45.15 (6a4a640fd8)

All required tools present.


In [ ]:
from enterprise_agents_on_foundry.setup import AzureContextError

try:
    check = check_subscription(settings.azure_subscription_id)
    print(check.message)
    print(f"Principal object id: {get_signed_in_principal_id()}")
except AzureContextError as error:
    print(f"Azure context unavailable: {error}")
    print("Run 'az login' and re-run this cell.")

### Model availability

The v0.1 work order asked for `gpt-5.1-mini` on Global Standard with 10K tokens per minute in `westus3`, and
required that an unavailable model produce a clear failure rather than a silent substitution.

`gpt-5.1-mini` is published in neither the regional catalog nor the quota list for `westus3`. The evidence and
the chosen alternative are in `docs/adr/004-model-deployment-selection.md`.

The check below is the same one the azd pre-provision hook runs. There is no fallback logic anywhere in the
codebase: a configured model that cannot be deployed stops provisioning.

In [14]:
from enterprise_agents_on_foundry.setup import ModelCatalogError

model_name = settings.azure_model_name or "gpt-5.4-mini"
model_version = settings.azure_model_version or "2026-03-17"

try:
    with time_operation(measurements, "model_catalog_check", category="setup"):
        availability = check_model_availability(
            location=settings.azure_location,
            model_name=model_name,
            model_version=model_version,
            sku_name="GlobalStandard",
            capacity=10,
        )

    reason = availability.failure_reason()
    if reason:
        print(f"BLOCKED: {reason}")
    else:
        remaining = (availability.quota_limit or 0) - (availability.quota_used or 0)
        print(f"{model_name} {model_version} is deployable. Remaining quota: {remaining:g}K TPM.")
except ModelCatalogError as error:
    print(f"Could not query the catalog: {error}")

gpt-5.4-mini 2026-03-17 is deployable. Remaining quota: 990K TPM.


## 9. What v0.1 will and will not provision

Will provision: the Foundry account and project, one model deployment, the Azure SQL server and serverless
AdventureWorksLT database, the Log Analytics workspace, Application Insights, Key Vault, the managed identity,
and five role assignments.

Will not provision: Azure Container Registry, Azure AI Search, Foundry IQ, long-term memory, private
networking, hosted agents, and external hosting. Each is parameterised and defaults to `false`, so a later
release flips a flag and adds a module rather than restructuring what exists.

No compute resource is created, because there is no application to host. No agent is created, because agent
construction belongs to later releases.

In [15]:
flags = {
    name.removeprefix("enable_"): getattr(settings, name)
    for name in type(settings).model_fields
    if name.startswith("enable_")
}

for name, enabled in sorted(flags.items()):
    print(f"{'on ' if enabled else 'off'}  {name}")

print()
print("Every optional capability is off, as v0.1 requires." if not any(flags.values()) else "A flag is on.")

off  azure_ai_search
off  container_registry
off  external_hosting
off  foundry_iq
off  hosted_agent
off  long_term_memory
off  private_networking

Every optional capability is off, as v0.1 requires.


### Provisioning

Nothing is created until the pre-provision check passes. It prints the topology, the region, and every billable
resource before anything happens, and exits non-zero if a tool is missing, the subscription does not match, or
the model cannot be deployed.

```console
azd auth login
azd env new dev
uv run python scripts/preprovision_check.py
azd provision
uv run --extra database python scripts/bootstrap_database.py
```

The bootstrap step needs the `database` extra for `pyodbc`, and `pyodbc` in turn needs the Microsoft ODBC Driver
18 for SQL Server, which is a system package rather than a Python dependency. The script checks for the driver
and prints installation guidance when it is absent.

Tear down with `azd down --purge`.

Resting cost is dominated by Azure SQL storage, since serverless compute auto-pauses and the Foundry account
bills only through model usage. The full cost summary is in `docs/architecture/v0.1-azure-foundation.md`.


## 10. Before and after measurements

Progress has to be measurable, otherwise modernization is an aesthetic claim.

Before: manual Conda setup, an unstructured `requirements.txt`, an embedded SQLite Chinook database, manually
assembled Azure infrastructure, classic Azure AI Foundry assumptions, and external hosting as the initial
deployment model.

After: Python 3.12 with a uv lockfile, a structured repository, notebook-guided setup, modular Bicep under the
azd lifecycle, an Azure SQL AdventureWorksLT foundation, a current Microsoft Foundry resource and project, a
monitoring foundation, and repeatable validation and cleanup.

In [ ]:
LEGACY_MANUAL_STEPS = [
    "Install Conda",
    "Create the Conda environment",
    "Activate the environment",
    "pip install -r requirements.txt",
    "Copy .env.example to .env",
    "Obtain an Azure OpenAI API key",
    "Paste the key into .env",
    "Locate chinook.db relative to the working directory",
]

CURRENT_STEPS = [
    "uv sync",
    "azd auth login",
    "azd provision",
    "uv run --extra database python scripts/bootstrap_database.py",
]

measurements.add("legacy_manual_setup_steps", len(LEGACY_MANUAL_STEPS), category="setup")
measurements.add("current_setup_steps", len(CURRENT_STEPS), category="setup")
measurements.add("legacy_runtime_dependencies", 31, category="repository")
measurements.add("current_runtime_dependencies", 4, category="repository")
measurements.add("legacy_committed_db_files", 5, category="repository")
measurements.add("current_committed_db_files", 0, category="repository")

for category, name, value, unit in measurements.as_rows():
    print(f"{category:<12} {name:<32} {value} {unit}")

setup        model_catalog_check              8272.6 ms
setup        legacy_manual_setup_steps        8 count
setup        current_setup_steps              4 count
repository   legacy_runtime_dependencies      31 count
repository   current_runtime_dependencies     4 count
repository   legacy_committed_db_files        5 count
repository   current_committed_db_files       0 count
setup        legacy_manual_setup_steps        8 count
setup        current_setup_steps              4 count
repository   legacy_runtime_dependencies      31 count
repository   current_runtime_dependencies     4 count
repository   legacy_committed_db_files        5 count
repository   current_committed_db_files       0 count


Database and provisioning measurements are captured against the live environment, so they run after
provisioning rather than here:

```console
uv run --extra database python scripts/bootstrap_database.py --measurements-out artifacts/v0.1-measurements.json
```

That records the schema count, the table count, the total approximate row count, the connection probe latency,
and the latency of the representative join and aggregate query. Row counts come from partition metadata rather
than `COUNT(*)`, which keeps the query cheap on a serverless database. `artifacts/` is excluded from version
control, so a measurement run never produces a commit.


## Verification

Six commands constitute the quality gate for this release.

```console
uv sync
uv run pytest
uv run ruff check .
uv run ruff format --check .
uv run mypy src scripts
az bicep build --file infra/main.bicep
```

## Next

`v0.2` rebuilds the Text-to-SQL agent on the current LangGraph line, replacing the hand-wired seven-node graph,
the fabricated tool call identifier, and the six-message rolling window. The roadmap is in `docs/roadmap.md`.